# 02 - Preprocessing

Preprocessing of the CIC-DDoS2019 dataset for binary classification (BENIGN vs DDoS).

Steps:
1. Load and merge raw CSV files (sampled)
2. Clean column names, remove duplicates
3. Handle infinite and missing values
4. Encode target variable (binary)
5. Drop identifier columns
6. Encode remaining categorical features
7. Feature selection — variance threshold + correlation filter
8. Save processed dataset

In [3]:
# Imports
import pandas as pd
import numpy as np
import glob
import os

In [4]:
# Configuration
DATA_DIR = "../data/raw/CSVs"
SAMPLE_ROWS = 1000  # rows per CSV file
RANDOM_STATE = 42

# Find all CSV files
csv_files = glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True)

print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f"  - {f}")

Found 18 CSV files
  - ../data/raw/CSVs\01-12\DrDoS_DNS.csv
  - ../data/raw/CSVs\01-12\DrDoS_LDAP.csv
  - ../data/raw/CSVs\01-12\DrDoS_MSSQL.csv
  - ../data/raw/CSVs\01-12\DrDoS_NetBIOS.csv
  - ../data/raw/CSVs\01-12\DrDoS_NTP.csv
  - ../data/raw/CSVs\01-12\DrDoS_SNMP.csv
  - ../data/raw/CSVs\01-12\DrDoS_SSDP.csv
  - ../data/raw/CSVs\01-12\DrDoS_UDP.csv
  - ../data/raw/CSVs\01-12\Syn.csv
  - ../data/raw/CSVs\01-12\TFTP.csv
  - ../data/raw/CSVs\01-12\UDPLag.csv
  - ../data/raw/CSVs\03-11\LDAP.csv
  - ../data/raw/CSVs\03-11\MSSQL.csv
  - ../data/raw/CSVs\03-11\NetBIOS.csv
  - ../data/raw/CSVs\03-11\Portmap.csv
  - ../data/raw/CSVs\03-11\Syn.csv
  - ../data/raw/CSVs\03-11\UDP.csv
  - ../data/raw/CSVs\03-11\UDPLag.csv


## Load Data (with sampling)


In [5]:
# Check if CSV files were found
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {DATA_DIR}")

In [6]:
def load_and_sample(filepath, n_rows=SAMPLE_ROWS, random_state=RANDOM_STATE):
    """Load first n rows from CSV to avoid memory issues."""
    df = pd.read_csv(filepath, nrows=n_rows, low_memory=False)
    return df

In [7]:
# Load all files
dfs = []

for f in csv_files:
    print(f"Loading {os.path.basename(f)}...")
    df_part = load_and_sample(f)
    dfs.append(df_part)
    print(f"  -> {len(df_part)} rows")

# Combine
df = pd.concat(dfs, ignore_index=True)

print(f"\nTotal: {len(df):,} rows, {df.shape[1]} columns")
df.head()

Loading DrDoS_DNS.csv...
  -> 1000 rows
Loading DrDoS_LDAP.csv...
  -> 1000 rows
Loading DrDoS_MSSQL.csv...
  -> 1000 rows
Loading DrDoS_NetBIOS.csv...
  -> 1000 rows
Loading DrDoS_NTP.csv...
  -> 1000 rows
Loading DrDoS_SNMP.csv...
  -> 1000 rows
Loading DrDoS_SSDP.csv...
  -> 1000 rows
Loading DrDoS_UDP.csv...
  -> 1000 rows
Loading Syn.csv...
  -> 1000 rows
Loading TFTP.csv...
  -> 1000 rows
Loading UDPLag.csv...
  -> 1000 rows
Loading LDAP.csv...
  -> 1000 rows
Loading MSSQL.csv...
  -> 1000 rows
Loading NetBIOS.csv...
  -> 1000 rows
Loading Portmap.csv...
  -> 1000 rows
Loading Syn.csv...
  -> 1000 rows
Loading UDP.csv...
  -> 1000 rows
Loading UDPLag.csv...
  -> 1000 rows

Total: 18,000 rows, 88 columns


,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,425,172.16.0.5-192.168.50.1-634-60495-17,172.16.0.5,634,192.168.50.1,60495,17,2018-12-01 10:51:39.813448,28415,97,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
1,430,172.16.0.5-192.168.50.1-60495-634-17,192.168.50.1,634,172.16.0.5,60495,17,2018-12-01 10:51:39.820842,2,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,DrDoS_DNS
2,1654,172.16.0.5-192.168.50.1-634-46391-17,172.16.0.5,634,192.168.50.1,46391,17,2018-12-01 10:51:39.852499,48549,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
3,2927,172.16.0.5-192.168.50.1-634-11894-17,172.16.0.5,634,192.168.50.1,11894,17,2018-12-01 10:51:39.890213,48337,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
4,694,172.16.0.5-192.168.50.1-634-27878-17,172.16.0.5,634,192.168.50.1,27878,17,2018-12-01 10:51:39.941151,32026,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS


## Basic Data Overiew


In [8]:
# Basic info
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes.value_counts())

print("\nShape:")
print(df.shape)

Columns:
['Unnamed: 0', 'Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Protocol', ' Timestamp', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Varian

## Clean Columns Names


In [9]:
# Clean column names
df.columns = df.columns.str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

Cleaned columns:
['Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag 

In [10]:
# Check target column
if "Label" not in df.columns:
    raise KeyError("Column 'Label' was not found in the dataset.")

## Remove Duplicates


In [11]:
# Remove duplicate rows
rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

print(f"Rows before removing duplicates: {rows_before:,}")
print(f"Rows after removing duplicates: {rows_after:,}")
print(f"Removed duplicates: {rows_before - rows_after:,}")

Rows before removing duplicates: 18,000
Rows after removing duplicates: 18,000
Removed duplicates: 0


## Handle Infinite Values


In [12]:
# Detect numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

print(f"Numeric columns: {len(numeric_cols)}")

Numeric columns: 82


In [13]:
# Count infinite values
inf_count = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values before cleaning: {inf_count}")

Infinite values before cleaning: 829


In [14]:
# Replace infinite values with NaN
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

inf_count_after = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values after cleaning: {inf_count_after}")

Infinite values after cleaning: 0


## Handle Missing Values


In [15]:
# Missing values before cleaning
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing_values)

Columns with missing values:
Flow Bytes/s      516
Flow Packets/s    516
dtype: int64


In [16]:
# Drop rows with missing values
rows_before = len(df)

df = df.dropna()

rows_after = len(df)

print(f"Rows before dropping missing values: {rows_before:,}")
print(f"Rows after dropping missing values: {rows_after:,}")
print(f"Removed rows: {rows_before - rows_after:,}")

Rows before dropping missing values: 18,000
Rows after dropping missing values: 17,484
Removed rows: 516


In [17]:
# Verify missing values
print(f"Total missing values after cleaning: {df.isna().sum().sum()}")

Total missing values after cleaning: 0


## Encode Target Variable


In [18]:
# Original label distribution
print("Original labels:")
print(df["Label"].value_counts())

Original labels:
Label
BENIGN           2004
NetBIOS          1907
Syn              1837
LDAP              998
UDP               998
DrDoS_SNMP        997
UDP-lag           997
DrDoS_LDAP        994
DrDoS_SSDP        994
DrDoS_MSSQL       981
DrDoS_NetBIOS     972
DrDoS_UDP         965
MSSQL             957
TFTP              884
DrDoS_DNS         633
DrDoS_NTP         283
Portmap            83
Name: count, dtype: int64


In [19]:
# Encode target for binary classification
# BENIGN -> 0
# DDoS attacks -> 1
df["target"] = df["Label"].apply(lambda label: 0 if label == "BENIGN" else 1)

print("Binary target distribution:")
print(df["target"].value_counts())

print("\nBinary target distribution (%):")
print(df["target"].value_counts(normalize=True) * 100)

Binary target distribution:
target
1    15480
0     2004
Name: count, dtype: int64

Binary target distribution (%):
target
1    88.538092
0    11.461908
Name: proportion, dtype: float64


In [20]:
# Drop original Label column
df = df.drop(columns=["Label"])

print(f"Columns after target encoding: {df.shape[1]}")

Columns after target encoding: 88


## Drop Identifier Columns

Identifier columns (IP addresses, Flow ID, Timestamp, SimillarHTTP) are removed — they are specific to the capture environment and would cause data leakage or overfitting in a general model.

In [21]:
# Drop identifier columns
identifier_cols = [
    "Unnamed: 0",
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "SimillarHTTP"
]

existing_identifier_cols = [col for col in identifier_cols if col in df.columns]

df = df.drop(columns=existing_identifier_cols)

print(f"Dropped identifier columns: {existing_identifier_cols}")
print(f"Shape after dropping identifiers: {df.shape}")

Dropped identifier columns: ['Unnamed: 0', 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'SimillarHTTP']
Shape after dropping identifiers: (17484, 82)


## Encode Categorical Features


In [22]:
# Detect categorical columns
categorical_cols = df.select_dtypes(include=["object","string"]).columns.tolist()

print(f"Categorical columns: {len(categorical_cols)}")
for col in categorical_cols:
    print(f"  - {col}")

Categorical columns: 0


In [23]:
# One-hot encode categorical columns
if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print("Categorical columns encoded.")
else:
    print("No categorical columns to encode.")

print(f"Shape after encoding: {df.shape}")

No categorical columns to encode.
Shape after encoding: (17484, 82)


In [24]:
# Convert boolean columns to integers
bool_cols = df.select_dtypes(include=["bool"]).columns

df[bool_cols] = df[bool_cols].astype(int)

print(f"Converted {len(bool_cols)} boolean columns to integers.")

Converted 0 boolean columns to integers.


## Feature Selection

Remove low-variance features and highly correlated features (|r| > 0.95) to reduce dimensionality and improve model generalization.

In [25]:
from sklearn.feature_selection import VarianceThreshold

target_col = df["target"]
features_df = df.drop(columns=["target"])

n_before = features_df.shape[1]
print(f"Features before variance threshold: {n_before}")

selector = VarianceThreshold(threshold=0.01)
selector.fit(features_df)
kept = features_df.columns[selector.get_support()]
features_df = features_df[kept]

print(f"Features after variance threshold:  {features_df.shape[1]}")
print(f"Removed {n_before - len(kept)} low-variance features")

Features before variance threshold: 81
Features after variance threshold:  68
Removed 13 low-variance features


In [26]:
# Remove highly correlated features (|r| > 0.95)
corr_matrix = features_df.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
high_corr = [c for c in upper.columns if any(upper[c] > 0.95)]

print(f"Features before correlation filter: {features_df.shape[1]}")
features_df = features_df.drop(columns=high_corr)
print(f"Features after correlation filter:  {features_df.shape[1]}")
print(f"Removed {len(high_corr)} highly correlated features")

# Reconstruct df with target
df = pd.concat([features_df, target_col], axis=1)

Features before correlation filter: 68
Features after correlation filter:  41
Removed 27 highly correlated features


## Normalization & Class Balancing

**Normalization** (`StandardScaler`) is applied in the modeling notebook (`03_modeling.ipynb`) to avoid data leakage — the scaler is fit only on training data.

**Class balancing** — due to class imbalance (~88.5% DDoS vs ~11.5% BENIGN) we use `class_weight="balanced"` built into the models (Random Forest, SVM). This automatically adjusts class weights inversely proportional to their frequency (cost-sensitive learning), which is preferred over SMOTE for network traffic data where synthetic samples may not be realistic.

## Final Check


In [27]:
# Check infinite values after preprocessing
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[numeric_cols]).sum().sum()

print(f"Infinite values after preprocessing: {inf_count}")

Infinite values after preprocessing: 0


In [28]:
# Final dataset check
print("Final shape:")
print(df.shape)

print("\nData types:")
print(df.dtypes.value_counts())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nTarget distribution:")
print(df["target"].value_counts())

Final shape:
(17484, 42)

Data types:
float64    25
int64      17
Name: count, dtype: int64

Missing values:
0

Target distribution:
target
1    15480
0     2004
Name: count, dtype: int64


In [29]:
df.head()

,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Std,...,Down/Up Ratio,Init_Win_bytes_forward,Init_Win_bytes_backward,min_seg_size_forward,Active Mean,Active Std,Active Max,Idle Std,Inbound,target
0,634,60495,17,28415,97,0,42680.0,0.0,440.0,0.0,...,0.0,-1,-1,-1,0.0,0.0,0.0,0.0,1,1
1,634,60495,17,2,2,0,880.0,0.0,440.0,0.0,...,0.0,-1,-1,-1,0.0,0.0,0.0,0.0,0,1
2,634,46391,17,48549,200,0,88000.0,0.0,440.0,0.0,...,0.0,-1,-1,-1,0.0,0.0,0.0,0.0,1,1
3,634,11894,17,48337,200,0,88000.0,0.0,440.0,0.0,...,0.0,-1,-1,-1,0.0,0.0,0.0,0.0,1,1
4,634,27878,17,32026,200,0,88000.0,0.0,440.0,0.0,...,0.0,-1,-1,-1,0.0,0.0,0.0,0.0,1,1


## Save Processed Data


In [30]:
# Save processed data
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/ddos_basic_preprocessed.csv"

df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows to {output_path}")

Saved 17,484 rows to ../data/processed/ddos_basic_preprocessed.csv
